# Pipeline Configuration Module

This notebook defines the centralized hyperparameter dataclass, file paths, hardware device configurations, and audio parameters for the audio deepfake detection pipeline.

In [1]:
import os
from dataclasses import dataclass
from pathlib import Path
import torch

@dataclass
class PipelineConfig:
    # Directory paths
    BASE_DIR: Path = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
    DATA_RAW_DIR: Path = BASE_DIR / "data" / "raw"
    DATA_PROCESSED_DIR: Path = BASE_DIR / "data" / "processed_clean"
    GRAPHS_DIR: Path = BASE_DIR / "graphs"
    EVALUATION_DIR: Path = BASE_DIR / "evaluation"
    MODELS_DIR: Path = BASE_DIR / "models"
    LOGS_DIR: Path = BASE_DIR / "logs"

    # Audio specifications
    SAMPLE_RATE: int = 16000          # 16 kHz required by standard Wav2Vec2
    TARGET_DURATION: float = 4.0      # 4 seconds fixed duration
    TARGET_SAMPLES: int = int(SAMPLE_RATE * TARGET_DURATION)  # 64,000 samples
    NUM_CHANNELS: int = 1             # Mono
    NORM_MAX_PEAK: float = 0.95       # Peak amplitude normalization ceiling

    # Dataset subsetting & balance (controls training set size for efficient training)
    MAX_SAMPLES_PER_CLASS: int = 1500  # 1,500 Real + 1,500 Fake = 3,000 samples
    RANDOM_SEED: int = 42

    # Split ratios (70% train, 15% validation, 15% test)
    TRAIN_SPLIT: float = 0.70
    VAL_SPLIT: float = 0.15
    TEST_SPLIT: float = 0.15

    # Model Architecture
    MODEL_NAME_OR_PATH: str = "facebook/wav2vec2-base"
    NUM_CLASSES: int = 2
    CLASS_LABELS: tuple = ("Real", "Fake")
    LABEL2ID: dict = None
    ID2LABEL: dict = None
    FREEZE_FEATURE_EXTRACTOR: bool = True
    UNFREEZE_TOP_TRANSFORMER_LAYERS: int = 2
    DROPOUT_RATE: float = 0.3
    HIDDEN_DIM: int = 256

    # Training Hyperparameters
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 0
    LEARNING_RATE: float = 1e-4
    BACKBONE_LR: float = 1e-5
    WEIGHT_DECAY: float = 1e-4
    NUM_EPOCHS: int = 5
    EARLY_STOPPING_PATIENCE: int = 2
    WARMUP_RATIO: float = 0.1
    USE_AMP: bool = True

    # Hardware Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Export
    BEST_MODEL_PATH: Path = MODELS_DIR / "best_wav2vec2_deepfake_model.pth"

    def __post_init__(self):
        self.LABEL2ID = {label: i for i, label in enumerate(self.CLASS_LABELS)}
        self.ID2LABEL = {i: label for i, label in enumerate(self.CLASS_LABELS)}
        for d in [self.GRAPHS_DIR, self.EVALUATION_DIR, self.MODELS_DIR, self.LOGS_DIR]:
            d.mkdir(parents=True, exist_ok=True)

config = PipelineConfig()


In [2]:
# Display Active Configuration
print(f"Device:             {config.DEVICE} ({torch.cuda.get_device_name(0) if config.DEVICE == 'cuda' else 'CPU'})")
print(f"Sampling Rate:      {config.SAMPLE_RATE} Hz")
print(f"Target Duration:    {config.TARGET_DURATION}s ({config.TARGET_SAMPLES:,} samples)")
print(f"Base Directory:     {config.BASE_DIR}")
print(f"Data Raw Dir:       {config.DATA_RAW_DIR}")
print(f"Model Architecture: {config.MODEL_NAME_OR_PATH}")


Device:             cuda (NVIDIA GeForce RTX 4060 Laptop GPU)
Sampling Rate:      16000 Hz
Target Duration:    4.0s (64,000 samples)
Base Directory:     e:\Deepfake
Data Raw Dir:       e:\Deepfake\data\raw
Model Architecture: facebook/wav2vec2-base
